# Alignment QC Notebook


# Import Libraries

In [ ]:
from tqdm import notebook
from datetime import datetime
from datetime import timedelta
import pandas as pd
import numpy as np

from datajoint.errors import DataJointError

# Terray Modules
from terray_datajoint.db import align
from terray_datajoint.db import exp
from terray_datajoint.db import decd
from terray_datajoint.db import extr
from terray_datajoint.db import imaging
from terray_core.utils.database import isin
from terray_notebooks.utils import panel_auto_qc as paq
from terray_notebooks.utils import database_guis_align as gui

# datajoint 0.13 renders restriction values into SQL with repr(), which under
# numpy 2 emits `np.int64(5)` instead of `5`. This notebook feeds ids fetched
# from the database back into queries, so restore the numpy 1 scalar reprs.
np.set_printoptions(legacy="1.25")

## Define which imaging runs you're interested in:

First we need to define which library runs we're interested in viewing.

### Here you can select:
**Run Name:** Consective keywords within the run names

For example, if you had a run name like SN7708-AA-x72-y82/Chip4 in your dataset:

* If you search for `SN7708-AA`, the query will match and return this run because `SN7708-AA` is a consecutive substring within the run_name.

* However, if you search for `SN7708 Chip4`, the query will not return this run because `SN7708 Chip4` is **NOT** a consecutive string in the run_name.

**exp_lgid:** A list of exp_lgids to look for

**Property Values:**
    * micrscope: A list of micrscopes to look for
    * exp_type: A list of exp_type to look for
    * spotted: A either "True" or "False"



## Define Which Imaging Runs You're Interested In

First, you need to define the library runs you're interested in viewing.

### Here You Can Select:

**Run Name:** Consecutive keywords within the run names.

For example, if you have a run name like `SN7708-AA-x72-y82/Chip4` in your dataset:

- If you search for `SN7708-AA`, the query will match and return this run because `SN7708-AA` is a consecutive substring within the `run_name`.
- However, if you search for `SN7708 Chip4`, the query will not return this run because `SN7708 Chip4` is **NOT** a consecutive string in the `run_name`.

**exp_lgid:** A list of `exp_lgid`s to look for. Make sure to keep "[]" if no experimental ids to query.

**Property Values:**
- **microscope:** A list of microscopes to look for.
- **exp_type:** A list of experiment types to look for.
- **spotted:** Either "True" or "False."

**Date Range** Select a date range to query. By default, the range includes the current date and time, looking back 10 days.
-  When selecting a date, times will default to the start of the day (midnight).

In [ ]:
query_params = paq.RunQueryParamsView()
query_params

## Query the databases

Running this cell will get the alignments for the imaging runs based on the variables you defined in the previous cell.

In [ ]:
# DON"T CHANGE ANYTHING HERE.
# Date Time
dt_start, dt_end = query_params.date_range
filter_query = filter_query = f"run_start BETWEEN '{dt_start}' AND '{dt_end}'"

# Get the Imaging Runs 
imaging_runs = exp.ImagingRuns * exp.LinkedRuns * exp.Experiments & filter_query

if query_params.exp_lgids:
    imaging_runs &= isin("exp_lgid", query_params.exp_lgids)

if query_params.run_name:
    imaging_runs &=  f"run_name LIKE '%{query_params.run_name}%'"
    
if len(imaging_runs) == 0:
    raise ValueError("No imaging runs found. Try adjusting date range, exp_lgids or run_name")

# Get the alignments for the imaging runs
# Use each run's latest alignment and latest photometry. Photometries are tagged
# per run, not per alignment, so without this every alignment of a run is paired
# with every tagged photometry of it. proj() drops date_tagged, which both tag
# tables have and DataJoint won't join on.
latest = {"vtag_name": "latest"}
latest_alignments = (align.TaggedAlignments & latest).proj("align_id")
latest_photometries = (extr.TaggedPhotometries & latest).proj("phot_id")
alignments = align.Alignments * latest_alignments & imaging_runs
alignments &= align.AlignmentAttachments & {'attach_name':'align_qc'}
alignments *= latest_photometries * imaging.QcFocus

# Create the Runs Dataframe to View

# Create the `runs_df` Based on Your Previous Query Parameters

The `runs_df` DataFrame is generated based on the query parameters you defined earlier. The columns in this DataFrame include:

- **`rawimg_id`:** The database ID for this imaging run.
- **`exp_lgid`:** The LabGuru experiment ID, also known as the "LG number."
- **`run_name`:** The name of this imaging run, derived from the original directory path on ImageStore.
- **`run_start`:** The timestamp for when this imaging run started.
- **`run_dur`:** The imaging duration in seconds.
- **`exp_title`:** The LabGuru experiment title (note that this may not reflect changes made in LabGuru after the Tiger Import).
- **`exp_owner`:** The LabGuru experiment owner.
- **`microscope`:** The microscope used to collect this imaging run.
- **`exp_type`:** The experiment type (entered during the Tiger import).
- **`spotted`:** The spotted status (entered during the Tiger import).
- **`align_id`:** The alignment ID, or blank if the alignment and alignment QC metrics haven't been computed yet.
- **`phot_id`:** The photo ID.
- **`median_focus_score_<ch_name>`:** The median focus score for the entire run for the specified channel `<ch_name>`.
- **`AF458_intensity`:** The aggregate (87.5th percentile) statistics of intensities of filled beads for channel AF458.


In [ ]:
# Columns to Keep
run_cols = {'rawimg_id',
            'exp_lgid',
            'run_name',
            'run_start',
            'run_dur',
            'exp_title',
            'exp_owner',
            }

align_cols = {"rawimg_id", "align_id",
              "phot_id",
              "median_focus_score_405",
              "median_focus_score_405",
              "median_focus_score_458",
             }

runs_df =  pd.DataFrame(imaging_runs.fetch(*run_cols, as_dict=True))
# run_dur comes back as Decimal, which the runs table (Bokeh) can't serialize
runs_df["run_dur"] = runs_df["run_dur"].astype(float)

# Initialize a list to collect property DataFrames
for (prop_name, prop_value) in query_params.property_values.items():     
    filtered_props = exp.RunProps & imaging_runs & {"prop_name": prop_name}
    
    # Further filter the table based on the provided prop_values
    if isinstance(prop_value, list):
        if len(prop_value) > 0:
            filtered_props &= isin("prop_value", prop_value)
    else:
        if prop_value != "":
            filtered_props &= {"prop_value": str(prop_value)}
            
    prop_dict = filtered_props.fetch("rawimg_id", "prop_value", as_dict=True)
    
    if not prop_dict:
        raise ValueError(f"No data found for property name '{prop_name}' with value '{prop_value}'. Please check the property name and value.")
    
    df =  pd.DataFrame(prop_dict).rename(columns={"prop_value": prop_name})
    runs_df = runs_df.merge(df, on="rawimg_id", how="inner")

if runs_df.empty:
    raise ValueError("The Dataframe is empty")

# Create an Alignments Dataframe
alignments &= isin("rawimg_id", runs_df.rawimg_id)
align_df = pd.DataFrame(alignments.fetch(*align_cols, as_dict=True))

# Add Intensities for All Channels to Alignment Dataframe
channel_intensities = pd.DataFrame(
    (alignments * (extr.PhotChannelStats & {"vtag_name": "latest", "bkg_sub": "matlab"}))
    .fetch("rawimg_id", "ch_name", "filled_p875", as_dict=True)
)

channel_intensities = channel_intensities.pivot(index="rawimg_id", columns="ch_name", values="filled_p875")
channel_intensities.columns = [f"{col}_intensity" for col in channel_intensities.columns]
channel_intensities.reset_index(inplace=True)
align_df = align_df.merge(channel_intensities, on='rawimg_id', how='inner')
    
# Add `align_df` to `runs_df`
runs_df = runs_df.merge(align_df, on='rawimg_id', how='inner')

## Add Metric Scores to the `runs_df`

This step will add the following columns to the `runs_df`:

- **`vignette_score_<ch_name>`:** A measurement of how uneven the vignette profile is, represented as a single number.
- **`%_well_separated_cntr`:** A metric that defines hybridization quality, indicating how well the flower plot lobes are generally separated.


In [ ]:
metric_scores_list = []
runs_df_copy = runs_df.copy()

for phot_id in runs_df["phot_id"]:
    # Add Vignette Scores
    metrics_scores = (extr.PhotVignette & {"phot_id": phot_id}).fetch(format="frame")
    metrics_scores = metrics_scores.pivot_table(index="phot_id", columns="ch_name", values="score").reset_index()
    metrics_scores.columns = ['phot_id'] + [f'vignette_score_{col}' for col in metrics_scores.columns[1:]]
    
    # Add hyb score
    try:
        hyb_qc_score = (decd.HybQcStats 
                        & {'qc_stat_name': '%_well_separated_cntr', 'baseline_phot_id': 0, "phot_id": phot_id}
                       ).fetch1("qc_stat_value")
    except DataJointError:
        hyb_qc_score = pd.NA
        
    metrics_scores["%_well_separated_cntr"] = hyb_qc_score  
    metric_scores_list.append(metrics_scores)

scores_df = pd.concat(metric_scores_list, ignore_index=True)
runs_df = runs_df.merge(scores_df, on="phot_id", how='left')
print("Total Runs Found:", len(runs_df))

## Add Misaligned Data

This counts the number of misaligned FOVs per imaging run, adding a `n_misaligned` column to the `runs_df` dataframe.

`n_misalgined` represents the number of FOVs with a pixel error greater than the threshold (misaligned_thresh) and are fully tracked in both dimensions across all channels.


<div style="border: 2px solid #FFA500; padding: 10px; border-radius: 5px; background-color: #FFF4E5;">
    <strong>⚠️ Warning:</strong> Downloading the alignment QC results can take about half a second per imaging run. Just FYI if you've got hundreds of imaging runs selected.
</div>


In [ ]:
metrics_list = []
for i in notebook.tqdm(runs_df.index, desc="Downloading Alignments"):
    align_id, phot_id = runs_df.loc[i, ["align_id", "phot_id"]]
    metrics = align.Alignments.for_id(align_id).get_qc_metrics()
    metrics_list.append(metrics)

In [ ]:
misaligned_thresh = 3
n_misaligned = []

for metrics in  metrics_list:
    is_misaligned = np.any(np.abs(metrics['total_px_err']) > misaligned_thresh, axis=0)
    is_misaligned &= np.all(metrics['tracked'], axis=0)  # Require that it be tracked in both axes
    is_misaligned = np.any(is_misaligned, axis=0)  # Reduce over the channel dimension
    n_misaligned.append(int(np.sum(is_misaligned)))
    
runs_df['n_misaligned'] = n_misaligned

# View the Runs Table

The following table allows for interactive exploration of the data.

## Customizing Column Display

You can adjust which columns are displayed and their order in the table through two simple lists:
`col_order`: Update this list with columns you'd like to explicitly display first and their preferred order.
```python
col_order = ['n_misaligned', 'median_focus_score_405', '%_well_separated_cntr']
```

`cols_to_hide`: Specify columns you'd like to completely hide from view. 
```python
cols_to_hide = []
```

### Note:
* Columns included in the default hidden list (default_cols_to_hide) will always remain hidden, in addition to any columns specified in your cols_to_hide list:
* The columns "exp_type" and "spotted" are always appended at the end of the table.


## Clicking A Row
* Selecting a row generates a black-and-white flower plot and metrics associated with that image.

## Green Slider
* Use this slider to filter the table based on the number of misaligned images (N Misaligned).
* The resulting filtered table is then used to display the alignment errors for all runs in the subsequent cell.


## Hyperlinks
* These links direct you to **Well Watcher**, providing detailed and useful troubleshooting views.

For more details on trouble shooting, see this slide [here](https://terraytherapeutics.sharepoint.com/:p:/r/sites/TerrayGeneral/_layouts/15/Doc.aspx?sourcedoc=%7BEE987885-221D-47DD-B238-E9F4F0C54F60%7D&file=decode_troubleshooting.pptx&action=edit&mobileredirect=true&DefaultItemOpen=1).


In [ ]:
default_cols_to_hide = ["exp_title", "exp_owner", "microscope", "align_id", "phot_id"]
print(f"Columns in the runs table:\n{'-' * 25}\n{list(runs_df.columns.copy().drop(default_cols_to_hide))}")

In [ ]:
# Update these lists to hide or manually order the columns in the runs table. 
# To do so, you can just copy the columns you want from the print statement in the previous cell
col_order = ['n_misaligned', 'median_focus_score_405', '%_well_separated_cntr']
cols_to_hide = []

##################################### Don't Touch anythin after this####################################
# Extend the list with default hidden columns
cols_to_hide += default_cols_to_hide

# Define last columns that should always appear at the end
last_cols = ["exp_type", "spotted"]

# Filter columns that are not in the hide list
df_cols = runs_df.columns.copy().drop(last_cols + cols_to_hide)

# Preserve user-defined column order while ensuring all remaining columns are included
col_order += list(df_cols[~df_cols.isin(col_order)])

# Ensure 'n_misaligned' is included if missing (needed for the slider)
if "n_misaligned" not in col_order:
    col_order.append("n_misaligned")

col_order += last_cols

In [ ]:
align_qc_figs = paq.AlignQcFigures(photometries_db=extr.Photometries)

runs_df.sort_values("run_start", inplace=True)
runs_table = paq.RunsTable(data=runs_df[col_order], align_qc_figures=align_qc_figs, cols_to_hide=cols_to_hide)
runs_table

# Alignment Error for All Channels: Vertical and Horizontal

This section provides heatmaps displaying the alignment error (in pixels) for the selected chip, as chosen from the dropdown box. A heatmap is generated for each channel, showing errors in both the vertical and horizontal directions.

The color in the heatmap represents the magnitude of the misalignment error in pixels, with each "block" corresponding to a Field of View (FOV). By hovering over a block, you can view the specific FOV number along with its corresponding horizontal and vertical pixel errors.


**Dropdown Info Includes:** Date | LG Number | Run Name | Raw Image ID

In [ ]:
align_heatmaps = paq.AlignAllHeatMaps(runs_df, metrics_list)
align_heatmaps

# Appendix

The following aren't used to often, but we can keep them here for future use on or for instances you might want to use them.

## Total Alignment Error

These heatmaps visualize the alignment error across all imaging runs.

- **The x-axis** in both heatmaps represents the FOV numbers (ranging from 1 to 792).
- **The y-axis** in both heatmaps corresponds to the imaging run index (which matches the "index" column from the previous RunsTable).
- **The color** indicates the magnitude of the misalignment error in pixels.
- **The top heatmap** provides a zoomed-in view of a selected region from the bottom heatmap.


In [ ]:
# filtered_df stays None until the table's slider filter first runs, which needs a
# browser to draw the table (not the case when microscope_toolbox runs this notebook).
filtered_runs_df = runs_df if runs_table.filtered_df is None else runs_table.filtered_df
total_align_err = paq.TotalAlignErr(runs_df=filtered_runs_df, metrics_list=metrics_list)
total_align_err